# Semana 38 · ¿Qué es la Vía Láctea y dónde estoy yo?

Este cuaderno acompaña a la sesión de proyecto de la semana 38: coordenadas galácticas, un mapa del cielo con estrellas reales, el vecindario del Sol en tres dimensiones y el error de Herschel.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. De ecuatoriales a galácticas

Escribimos cada dirección como un vector unitario y lo giramos con la matriz $T$ (la de la sesión 1, con más decimales).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = np.array([[-0.0548755604, -0.8734370902, -0.4838350155],
              [ 0.4941094279, -0.4448296300,  0.7469822445],
              [-0.8676661490, -0.1980763734,  0.4559837762]])

def ecuatorial_a_galactica(ar_grados, dec_grados):
    ar, dec = np.radians(ar_grados), np.radians(dec_grados)
    v = np.array([np.cos(dec) * np.cos(ar), np.cos(dec) * np.sin(ar), np.sin(dec)])
    x, y, z = T @ v
    return np.degrees(np.arctan2(y, x)) % 360, np.degrees(np.arcsin(z))

for nombre, ar, dec in [("centro galáctico (Sgr A*)", 266.417, -29.008), ("polo norte celeste", 0, 90),
                        ("Deneb", 310.358, 45.280), ("Betelgeuse", 88.793, 7.407), ("Vega", 279.235, 38.784)]:
    l, b = ecuatorial_a_galactica(ar, dec)
    print(f"{nombre:<28} ℓ = {l:6.1f}°   b = {b:6.1f}°")
print("¿Es T una rotación? T·Tᵀ =\n", np.round(T @ T.T, 6))

$T\,T^\top$ es la matriz identidad: las filas son vectores unitarios y perpendiculares entre sí. Es lo que define a una matriz de rotación (semana 19).

## 2. El cielo en coordenadas galácticas

Cargamos el catálogo HYG. Sus columnas `x`, `y`, `z` son posiciones en pársecs en coordenadas ecuatoriales; al girarlas con $T$ obtenemos posiciones galácticas, con el eje $x$ hacia el centro de la galaxia y el $z$ hacia el polo norte galáctico.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/alabcar/hola-universo/main/cuadernos/datos/estrellas_hyg.csv"
locales = ["cuadernos/datos/estrellas_hyg.csv", "../datos/estrellas_hyg.csv", "datos/estrellas_hyg.csv"]
ruta = next((r for r in locales if os.path.exists(r)), URL)
estrellas = pd.read_csv(ruta)
print(f"{len(estrellas)} estrellas leídas de {ruta}")

In [ ]:
est = estrellas[estrellas.dist_pc > 0].copy()
xyz_gal = T @ est[["x", "y", "z"]].to_numpy().T
est["xg"], est["yg"], est["zg"] = xyz_gal
est["l"] = np.degrees(np.arctan2(est.yg, est.xg)) % 360
est["b"] = np.degrees(np.arcsin(est.zg / est.dist_pc))

def a_mollweide(l_grados):
    """Longitud galáctica a radianes de -π a π, creciendo hacia la izquierda, como en los mapas astronómicos."""
    return -np.radians(((l_grados + 180) % 360) - 180)

cercanas = est[est.dist_pc < 25]
lejanas = est[(est.dist_pc > 150) & (est.mag < 4)]
fig = plt.figure(figsize=(12, 6.5))
ax = fig.add_subplot(projection="mollweide")
ax.scatter(a_mollweide(cercanas.l), np.radians(cercanas.b), s=4, c="tab:orange", label=f"a menos de 25 pc ({len(cercanas)})")
ax.scatter(a_mollweide(lejanas.l), np.radians(lejanas.b), s=10, c="tab:blue", label=f"brillantes a más de 150 pc ({len(lejanas)})")
ax.set_xticklabels(["150°", "120°", "90°", "60°", "30°", "0°", "330°", "300°", "270°", "240°", "210°"])
ax.grid(alpha=0.4); ax.legend(loc="lower right", fontsize=8)
ax.set_title("El cielo en coordenadas galácticas (centro galáctico en el medio)")
plt.show()

Las estrellas cercanas se reparten por todo el cielo: el vecindario del Sol es más o menos una esfera. Las estrellas luminosas y lejanas se concentran cerca de $b = 0$: el ecuador galáctico, el plano del disco. Es la Vía Láctea apareciendo en los datos.

## 3. El vecindario del Sol en tres dimensiones

In [ ]:
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")
color = cercanas.b_v.clip(-0.3, 2.0)
ax.scatter(cercanas.xg, cercanas.yg, cercanas.zg, s=4, c=color, cmap="RdYlBu_r")
ax.scatter([0], [0], [0], s=60, c="gold", edgecolors="k", label="Sol")
for nombre in ["Sirius", "Procyon", "Altair", "Vega", "Rigil Kentaurus", "Barnard's Star"]:
    f = cercanas[cercanas.nombre == nombre]
    if len(f):
        ax.text(f.xg.iloc[0], f.yg.iloc[0], f.zg.iloc[0], nombre, fontsize=7)
ax.set_xlabel("hacia el centro galáctico (pc)"); ax.set_ylabel("sentido del giro (pc)"); ax.set_zlabel("hacia el polo norte galáctico (pc)")
ax.legend(); ax.set_title("Estrellas a menos de 25 pc"); plt.show()

En Colab puedes girar la vista cambiando `ax.view_init(elev, azim)` antes de `plt.show()`. A esta escala, el disco de la galaxia (de unos cientos de pársecs de espesor) ni se nota: el vecindario parece uniforme.

## 4. El error de Herschel

Un modelo de juguete: una galaxia plana (un disco de 15 kpc de radio, en dos dimensiones) con dos millones de estrellas, más densa hacia el centro. El Sol está a 8,2 kpc del centro. Contamos, en cada dirección del plano, las estrellas más brillantes que una magnitud límite, primero sin polvo y luego con una extinción de 1,5 magnitudes por kpc. Todas las estrellas tienen magnitudes absolutas al azar entre −2 y 5. **Es una simulación sencilla**, no un modelo real de la galaxia.

In [ ]:
rng = np.random.default_rng(38)
n = 2_000_000
r = rng.exponential(3.5, n)                          # kpc, más densa hacia el centro
r = r[r < 15]
fi = rng.uniform(0, 2 * np.pi, r.size)
x, y = r * np.cos(fi), r * np.sin(fi)
sol = np.array([-8.2, 0.0])                          # el centro está en el origen
dx, dy = x - sol[0], y - sol[1]
d_kpc = np.hypot(dx, dy)
direccion = np.degrees(np.arctan2(dy, dx)) % 360     # 0° = hacia el centro galáctico
M = rng.uniform(-2, 5, r.size)

fig, axs = plt.subplots(1, 2, figsize=(12, 5.5), subplot_kw={"projection": "polar"})
for ax, a, titulo in [(axs[0], 0.0, "sin polvo"), (axs[1], 1.5, "con polvo (1,5 mag/kpc)")]:
    m = M + 5 * np.log10(d_kpc * 1000 / 10) + a * d_kpc
    visibles = m < 11
    cuentas, bordes = np.histogram(direccion[visibles], bins=36, range=(0, 360))
    centros = np.radians(bordes[:-1] + 5)
    ax.bar(centros, cuentas, width=np.radians(10), color="tab:blue", alpha=0.7)
    ax.set_title(f"{titulo}\n(0° = dirección del centro)")
plt.tight_layout(); plt.show()

Sin polvo, se ven muchas más estrellas hacia el centro de la galaxia: el Sol está claramente descentrado. Con polvo, solo se ven las estrellas de unos pocos kilopársecs alrededor, y los recuentos se vuelven casi iguales en todas direcciones: parece que el Sol está en el centro. Es exactamente la trampa en la que cayeron Herschel y Kapteyn.

## 5. Experimenta

1. Calcula las coordenadas galácticas de las Pléyades (AR 56,75°, Dec +24,12°) y de la nebulosa de Orión (AR 83,82°, Dec −5,39°). ¿Están cerca del plano galáctico?
2. En el mapa del cielo, dibuja solo las estrellas de tipo espectral O y B (las jóvenes y calientes). ¿Dónde se concentran?
3. En la simulación de Herschel, prueba con una extinción de 0,5 y de 3 mag/kpc, y con una magnitud límite de 6 (a simple vista). ¿Cuándo «parece» el Sol central?

In [ ]:
# Tu experimento aquí